# PG-MoE — IMU Expert 训练

这个 notebook 训练 PG-MoE 的 **IMU expert**（Xiaoyang 负责的部分）。

**目标：**
1. 用 Deep 1D-CNN + residual 训练 IMU 单模态分类器
2. 准确率超过 midterm baseline 的 **67.9%**
3. 保存 encoder 权重 `imu_expert.pt`，给联合 PG-MoE 训练加载

每个 cell 之前都有一段 markdown 解释。**一个一个 cell 跑**，理解了再往下走。

---

## 1. 挂载 Google Drive

数据集 (UTD-MHAD) 和模型权重都在 Drive 上读写。这样即使 Colab session 断了，数据和权重也不会丢。

运行后会让你授权 Google 账号，照着提示点确认就行。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 路径和超参数

- `DATA_ROOT`：UTD-MHAD 数据集根目录（里面有 `Inertial/` 子目录）
- `SAVE_DIR`：模型权重保存目录，跑完会有 `imu_classifier_best.pt` 和 `imu_expert.pt` 两个文件
- `IMU_LEN = 192`：把每段 IMU 截/补齐到 192 个时间步（约 1.9 秒 @ 100 Hz）
- `D_MODEL = 256`：和 `pgmoe_plan.md` 第三节约定一致，方便和 Hang 的 vision token 对接

**如果你的 Drive 路径不一样，改 `DATA_ROOT` 这一行。**

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/utd_mhad"
SAVE_DIR  = "/content/drive/MyDrive/pgmoe_ckpt"

IMU_LEN     = 192
NUM_CLASSES = 27
D_MODEL     = 256
EPOCHS      = 80
BATCH_SIZE  = 32
LR          = 1e-3
WEIGHT_DECAY = 1e-4

import os
os.makedirs(SAVE_DIR, exist_ok=True)
print("Data root:", DATA_ROOT)
print("Save dir :", SAVE_DIR)

## 3. 导入库

- `scipy.io` 用来读 `.mat` 文件（UTD-MHAD 的原始格式）
- `torch`、`torch.nn`：PyTorch 主体
- `sklearn.metrics`：算 accuracy / confusion matrix
- `matplotlib`：画训练曲线和混淆矩阵

最后打印 GPU 是否可用。Colab 免费层应该是 T4 或者 V100。

In [ ]:
import os, time
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Torch :", torch.__version__)
if device.type == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))

## 4. 看一眼原始数据

UTD-MHAD 的 IMU 文件名格式：**`aA_sS_tT_inertial.mat`**
- `A` = action label (1–27)
- `S` = subject id (1–8)
- `T` = trial number

每个 `.mat` 里有个 `d_iner` 数组，shape 是 `(timesteps, 6)`。6 个通道是 **[acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z]**。

注意：**timesteps 长度不固定**——每个动作执行的时间不一样。我们后面会统一截/补齐到 192。

In [ ]:
inertial_folder = os.path.join(DATA_ROOT, "Inertial")
files = sorted([f for f in os.listdir(inertial_folder) if f.endswith("_inertial.mat")])
print(f"Total IMU files: {len(files)}")
print("First 5 files :", files[:5])

sample = sio.loadmat(os.path.join(inertial_folder, files[0]))
print("\nKeys in .mat   :", [k for k in sample.keys() if not k.startswith("_")])
print("d_iner shape   :", sample["d_iner"].shape)
print("d_iner dtype   :", sample["d_iner"].dtype)
print("First 3 rows   :\n", sample["d_iner"][:3])

## 5. 解析文件名 + 划分 train / test

UTD-MHAD 标准做法是 **subject-based split**：
- 奇数 subjects `{1, 3, 5, 7}` → 训练
- 偶数 subjects `{2, 4, 6, 8}` → 测试

**为什么这样切？** 保证测试集里的人在训练时**完全没见过**，避免模型记住某个人的运动风格而虚高准确率。

In [ ]:
def parse_filename(fname):
    parts = fname.split("_")
    action  = int(parts[0][1:])  # 'a3'  -> 3
    subject = int(parts[1][1:])  # 's5'  -> 5
    trial   = int(parts[2][1:])  # 't2'  -> 2
    return action, subject, trial

print(parse_filename("a3_s5_t2_inertial.mat"))   # 期望 (3, 5, 2)

## 6. 构造 Dataset 类

PyTorch 的 `Dataset` 只需要两个方法：`__len__` 和 `__getitem__`。

几个工程细节：
- **长度统一**：IMU 实际长度不固定，短的补 0 到 192，长的截到 192
- **转置**：`d_iner` 是 `(T, 6)`，但 `Conv1d` 要 `(channels, length)` 即 `(6, T)`，所以 `.T`
- **label 减 1**：数据集里类别是 1–27，但 `CrossEntropyLoss` 要 0–26

In [ ]:
TRAIN_SUBJECTS = {1, 3, 5, 7}
TEST_SUBJECTS  = {2, 4, 6, 8}

class IMUDataset(Dataset):
    def __init__(self, data_root, train=True):
        self.samples = []
        allowed = TRAIN_SUBJECTS if train else TEST_SUBJECTS
        folder = os.path.join(data_root, "Inertial")
        for fname in sorted(os.listdir(folder)):
            if not fname.endswith("_inertial.mat"):
                continue
            action, subject, _ = parse_filename(fname)
            if subject not in allowed:
                continue
            self.samples.append((os.path.join(folder, fname), action - 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fpath, label = self.samples[idx]
        data = sio.loadmat(fpath)["d_iner"].astype(np.float32)
        if data.shape[0] < IMU_LEN:
            pad = np.zeros((IMU_LEN - data.shape[0], 6), np.float32)
            data = np.concatenate([data, pad], axis=0)
        x = torch.from_numpy(data[:IMU_LEN]).T.contiguous()   # (6, 192)
        return x, label

train_ds = IMUDataset(DATA_ROOT, train=True)
test_ds  = IMUDataset(DATA_ROOT, train=False)
print(f"Train: {len(train_ds)} samples")
print(f"Test : {len(test_ds)} samples")

x0, y0 = train_ds[0]
print(f"\nFirst sample x shape: {tuple(x0.shape)}, label: {y0}")

## 7. 定义 `ResidualBlock1D`

残差块的核心思想：**`y = F(x) + x`**，让网络学的是"残差"，而不是从零学整个映射。

**为什么要残差：**
- 深层网络容易出现 degradation（更深反而更差）
- `F(x)=0` 时 `y=x`，恒等映射是一个合理的初始状态
- 梯度能通过 shortcut 直接回传，缓解梯度消失

**Shortcut 路径**有个判断：当输入输出**通道数不一样**或**长度不一样**（stride≠1）时，没法直接 `+`，得用一个 1×1 卷积把 `x` 调整到匹配的形状。这就是 ResNet 论文里的 "projection shortcut"。

In [ ]:
class ResidualBlock1D(nn.Module):
    def __init__(self, in_c, out_c, kernel=5, stride=1):
        super().__init__()
        pad = kernel // 2
        self.conv = nn.Sequential(
            nn.Conv1d(in_c, out_c, kernel, stride=stride, padding=pad),
            nn.BatchNorm1d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_c, out_c, kernel, stride=1, padding=pad),
            nn.BatchNorm1d(out_c),
        )
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_c, out_c, 1, stride=stride),
                nn.BatchNorm1d(out_c),
            )
        else:
            self.shortcut = nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv(x) + self.shortcut(x))

# 验证 shape
block = ResidualBlock1D(64, 128, kernel=5, stride=2)
dummy = torch.randn(2, 64, 96)
print("Input :", tuple(dummy.shape))
print("Output:", tuple(block(dummy).shape))   # 期望 (2, 128, 48)

## 8. 定义 `IMUExpert`（核心模型）

四段下采样，每段长度减半、通道翻倍：

| 层      | kernel | stride | 输出 shape    | 作用                                |
|---------|--------|--------|---------------|-------------------------------------|
| stem    | 7      | 2      | (64, 96)      | 大 kernel 看长时程，6 通道升到 64   |
| block1  | 5      | 2      | (128, 48)     | 抽局部运动模式                       |
| block2  | 5      | 2      | (256, 24)     | 更深特征                             |
| block3  | 3      | 2      | (256, 12)     | 精细调整，到达 T_i=12                |

最后 `transpose(1, 2)` 把 `(B, C, L)` 转成 `(B, L, C)`，因为 cross-attention 习惯 sequence-first 的格式。

**关键：不做 global pool！** 输出保留 12 个时间 token。这是和 midterm `IMUNet` 最大的区别。

In [ ]:
class IMUExpert(nn.Module):
    def __init__(self, d_model=256, in_channels=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
        )
        self.block1 = ResidualBlock1D(64, 128, kernel=5, stride=2)
        self.block2 = ResidualBlock1D(128, 256, kernel=5, stride=2)
        self.block3 = ResidualBlock1D(256, d_model, kernel=3, stride=2)

    def forward(self, x):
        x = self.stem(x)      # (B, 64, 96)
        x = self.block1(x)    # (B, 128, 48)
        x = self.block2(x)    # (B, 256, 24)
        x = self.block3(x)    # (B, 256, 12)
        return x.transpose(1, 2)   # (B, 12, 256)

encoder = IMUExpert()
dummy = torch.randn(4, 6, 192)
out = encoder(dummy)
print("Encoder input :", tuple(dummy.shape))
print("Encoder output:", tuple(out.shape))    # 期望 (4, 12, 256)
print("Encoder params:", f"{sum(p.numel() for p in encoder.parameters()):,}")

## 9. 定义 `IMUClassifier`（临时分类头）

**这里出现了 mean pool**——但和 midterm `AdaptiveAvgPool1d(1)` 的区别是：
- 这个 pool 只在**临时的分类头**里做
- 目的：让我们能算单模态准确率，sanity check
- 联合训练时**只加载 `self.encoder`**，把 head 整个扔掉
- 所以联合训练时时序信息完全保留

`LayerNorm` 在 pool 后做归一化，让 logits 数值更稳。这是 transformer 时代的标配。

In [ ]:
class IMUClassifier(nn.Module):
    def __init__(self, num_classes=27, d_model=256):
        super().__init__()
        self.encoder = IMUExpert(d_model=d_model)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        tokens = self.encoder(x)          # (B, 12, 256)
        pooled = tokens.mean(dim=1)       # (B, 256)  ← 临时 pool，仅本头用
        return self.head(self.norm(pooled))

model = IMUClassifier().to(device)
dummy = torch.randn(4, 6, 192).to(device)
print("Classifier output:", tuple(model(dummy).shape))   # 期望 (4, 27)
print("Total params     :", f"{sum(p.numel() for p in model.parameters()):,}")

## 10. DataLoader、Optimizer、LR Scheduler

- **DataLoader**：`shuffle=True` for train（避免学到数据顺序）；`num_workers=2` 让数据加载并行；`pin_memory=True` 让 CPU→GPU 拷贝更快
- **AdamW**（不是 Adam）：weight_decay 实现更正确（Loshchilov 2019）
- **CosineAnnealingLR**：lr 从 1e-3 余弦衰减到 0；训练后期 lr 小，能精细调整
- **CrossEntropyLoss**：标准多类分类损失

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

## 11. 评估函数

`model.eval()` 会：
- 关掉 BatchNorm 的 running stats 更新（用训练时累积的均值/方差）
- 关掉 Dropout（如果有）

`torch.no_grad()` 不存梯度计算图，节省显存。

In [ ]:
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(device))
            preds.extend(logits.argmax(1).cpu().numpy())
            labels.extend(y.numpy())
    return accuracy_score(labels, preds), np.array(preds), np.array(labels)

## 12. 训练循环

每个 epoch：
1. 训练一轮（forward + backward + step）
2. LR scheduler 步进
3. 在测试集上算 accuracy
4. 如果是最佳，保存权重（两个文件：完整 classifier、仅 encoder）

**预期：**
- 前 5–10 epoch 应该突破 **50%**（说明训练没崩）
- 最终 best 应 **> 67.9%**（midterm baseline），否则我们设计有问题
- 如果最终 > **75%**，那"深 1D-CNN + 保留时序 + 残差"假设就强力地验证了

GPU 上 80 epoch 大概 5–10 分钟。每行末尾的 `*` 表示该 epoch 是当前最佳。

In [ ]:
best_acc, best_epoch = 0.0, -1
history = {"loss": [], "acc": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    scheduler.step()
    train_loss /= len(train_ds)

    acc, _, _ = evaluate(model, test_loader)
    history["loss"].append(train_loss)
    history["acc"].append(acc)

    flag = ""
    if acc > best_acc:
        best_acc, best_epoch = acc, epoch
        torch.save(model.state_dict(),
                   os.path.join(SAVE_DIR, "imu_classifier_best.pt"))
        torch.save(model.encoder.state_dict(),
                   os.path.join(SAVE_DIR, "imu_expert.pt"))
        flag = "  *"
    print(f"Epoch {epoch:3d} | loss {train_loss:.4f} | "
          f"acc {acc:.4f} | {time.time()-t0:.1f}s{flag}")

print(f"\nBest: {best_acc:.4f} ({best_acc*100:.2f}%) at epoch {best_epoch}")
print(f"midterm baseline: 67.90%   delta: {(best_acc-0.679)*100:+.2f}%")

## 13. 训练曲线

看两个信号：
- **Loss 下降但 acc 不涨** → 在过拟合训练集，可能需要更多正则（dropout、mixup、更小模型）
- **Acc 在最后几个 epoch 还在涨** → epochs 不够，可以加到 100+
- **Acc 早期就 plateau** → lr 可能太大或太小，或者模型容量不够

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["loss"]); axes[0].set_title("Train loss"); axes[0].set_xlabel("epoch")
axes[1].plot(history["acc"]);  axes[1].set_title("Test accuracy"); axes[1].set_xlabel("epoch")
axes[1].axhline(0.679, ls="--", color="gray", label="midterm 67.9%")
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "training_curve.png"), dpi=120)
plt.show()

## 14. 每类准确率 + 混淆矩阵

**为什么重要：** 哪些动作识别得好、哪些容易混淆？这对 PG-MoE 后面的 failure analysis 非常有用——

比如如果 IMU 在 "throw" 和 "basketball" 上混淆严重，那正好说明 vision 应该在这些类上承担更多权重，phase arbitrator 的 α 在这些动作上应该偏大。

如果某些类样本数特别少（<5），那 per-class acc 噪声大，不用太当真。

In [ ]:
model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "imu_classifier_best.pt")))
_, preds, labels = evaluate(model, test_loader)

print("Per-class accuracy (sorted ascending — worst first):")
rows = []
for c in range(NUM_CLASSES):
    mask = labels == c
    if mask.sum() == 0:
        continue
    rows.append((c, (preds[mask] == c).mean(), int(mask.sum())))
rows.sort(key=lambda r: r[1])
for c, acc_c, n in rows:
    bar = "#" * int(acc_c * 30)
    print(f"  class {c:2d} | acc {acc_c:.3f} | n={n:3d} | {bar}")

In [ ]:
cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.title("Confusion matrix (IMU expert)")
plt.xlabel("predicted")
plt.ylabel("true")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=120)
plt.show()

## 15. 总结 & 下一步

**跑完应该有的 4 个产物：**
1. `imu_classifier_best.pt` — 完整模型（含分类 head），用来复现这个准确率
2. `imu_expert.pt` — **只有 encoder**，联合训练时 PG-MoE 加载这个
3. `training_curve.png` — 训练曲线
4. `confusion_matrix.png` — 混淆矩阵

**还要做的事：**
1. 把 best acc 数字填到 `project/final/devlog.md` 的 "Xiaoyang 的进度" 那一节
2. 把两张 png 复制到 `project/final/figures/` 留作 report 用
3. 告诉 Hang 你的 IMU expert 输出是 `(B, 12, 256)`，权重在 Drive 的 `pgmoe_ckpt/imu_expert.pt`

**下一步：** 写 `phase_arbitrator.py` —— 从 IMU 原始信号抽物理特征（加速度模、二阶导、能量率） → Phase Encoder → α(t)。**这才是整篇 paper 的 novelty。**